# Database Initialization Script

## What This Script Does

Creates a SQLite database (`churn.db`) with three tables:

### 1. `raw_data`
- Stores all 7,043 rows from `telco_churn.csv`
- Contains original customer data including Churn labels
- Used as the historical training dataset

### 2. `new_data`
- Takes 5 random rows from the original dataset
- Adds columns for predictions (`prediction`, `probability`, `churn_label`, `timestamp`)
- Used to simulate new customer predictions awaiting labeling

### 3. `api_keys`
- Stores API key hashes for authentication
- Currently has placeholder values (`placeholder_user_hash`, `placeholder_admin_hash`)
- Must be updated on the server with real hashed keys

## Why We Need This

- **Separation of concerns**: Raw data, new predictions, and keys are in different tables
- **Simulates production**: New customers go to `new_data`, get predictions, then can be labeled
- **Security**: Keys are hashed before storage (placeholders will be replaced)

## How to Use

1. Run this script once to create the database
2. On the server, run `update_keys.py` to enter your real API keys
3. The FastAPI app will use this database for all operations

## Output

- Database location: `app/data/DB/churn.db`
- 3 tables created with appropriate schemas
- Ready for use with the Churn Prediction API

In [2]:
"""
Script to create SQLite database with:
1. raw_data - all original telco churn data
2. new_data - 200 random rows with predictions and modified labels
3. api_keys - API keys (will be entered at startup)

Run this script ONCE to initialize the database.
"""
import sys
import random
from datetime import datetime, timedelta
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path
import cloudpickle
import hashlib



# Paths
DB_PATH = Path("../app/data/DB/churn.db")
CSV_PATH = Path("../data/raw/telco_churn.csv")
MODEL_PATH = Path("../models/full_churn_pipeline_cloud.pkl")


def load_model():
    """Load the trained model"""
    
    with open(MODEL_PATH, 'rb') as f:
        model = cloudpickle.load(f)
    print(f"✅ Model loaded from {MODEL_PATH}")
    return model


def make_predictions(model, df):
    """Make predictions for all rows in dataframe"""
    # Features needed for prediction
    feature_columns = [
        'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
        'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
        'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
        'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
        'MonthlyCharges', 'TotalCharges'
    ]
    
    X = df[feature_columns].copy()
    
    # Make predictions
    predictions = model.predict(X)
    probabilities = model.predict_proba(X)[:, 1]
    
    return predictions, probabilities


def create_database():
    """Create SQLite database with all three tables."""
    
    # Create data directory if it doesn't exist
    DB_PATH.parent.mkdir(parents=True, exist_ok=True)
    
    # Connect to database
    conn = sqlite3.connect(str(DB_PATH))
    cursor = conn.cursor()
    
    # =========================================================
    # TABLE 1: raw_data (all original data)
    # =========================================================
    print("\n" + "="*60)
    print("STEP 1: Loading raw data")
    print("="*60)
    
    df_raw = pd.read_csv(CSV_PATH)
    print(f"✅ Loaded {len(df_raw)} rows from {CSV_PATH}")
    
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS raw_data (
            customerID TEXT PRIMARY KEY,
            gender TEXT,
            SeniorCitizen INTEGER,
            Partner TEXT,
            Dependents TEXT,
            tenure INTEGER,
            PhoneService TEXT,
            MultipleLines TEXT,
            InternetService TEXT,
            OnlineSecurity TEXT,
            OnlineBackup TEXT,
            DeviceProtection TEXT,
            TechSupport TEXT,
            StreamingTV TEXT,
            StreamingMovies TEXT,
            Contract TEXT,
            PaperlessBilling TEXT,
            PaymentMethod TEXT,
            MonthlyCharges REAL,
            TotalCharges REAL,
            Churn TEXT
        )
    ''')
    
    df_raw.to_sql('raw_data', conn, if_exists='replace', index=False)
    print(f"✅ Table 'raw_data' created with {len(df_raw)} rows")
    
        # =========================================================
    # TABLE 2: new_data (200 random rows with predictions)
    # =========================================================
    print("\n" + "="*60)
    print("STEP 2: Creating new_data with 200 random rows")
    print("="*60)
    
    # Take 200 random rows
    df_new = df_raw.sample(n=200, random_state=42).copy()
    df_new = df_raw.sample(n=200, random_state=42).reset_index(drop=True).copy()
    print(f"✅ Selected {len(df_new)} random rows")
    print(f"   Shape before predictions: {df_new.shape}")
    
    # Load model and make predictions
    print("📦 Loading model and making predictions...")
    model = load_model()
    predictions, probabilities = make_predictions(model, df_new)
    
    # Add prediction columns (THIS ADDS COLUMNS, NOT ROWS!)
    df_new['prediction'] = predictions
    df_new['probability'] = probabilities
    df_new['created_at'] = datetime.now().isoformat()
    
    print(f"   Shape after adding predictions: {df_new.shape}")  # Должно быть (200, X)
    
    # =========================================================
    # STEP 3: Set labels for ALL 200 rows
    # =========================================================
    print("\n" + "="*60)
    print("STEP 3: Setting labels for all 200 rows")
    print("="*60)
    
    # Copy original Churn to churn_label (THIS ADDS A COLUMN, NOT ROWS!)
    df_new['churn_label'] = df_new['Churn'].copy()
    
    current_time = datetime.now()
    
    # Set label_timestamp for ALL rows (THIS ADDS A COLUMN, NOT ROWS!)
    df_new['label_timestamp'] = [
        (current_time + timedelta(hours=random.uniform(1, 3))).isoformat()
        for _ in range(len(df_new))  # ← range(len(df_new))
    ]
    
    print(f"   Shape after adding label columns: {df_new.shape}")  
    
    # Get 20 random indices to FLIP the label
    random_indices = random.sample(range(len(df_new)), 20)
    
    for idx in random_indices:
        original_label = df_new.iloc[idx]['churn_label']
        new_label = 'No' if original_label == 'Yes' else 'Yes'
        
        df_new.loc[idx, 'churn_label'] = new_label
        
    
    print(f"✅ Labels set for all {len(df_new)} rows")
    print(f"   - {len(random_indices)} rows have FLIPPED labels")
    print(f"   - Final shape: {df_new.shape}")
    
    # Remove original Churn column
    df_new = df_new.drop(columns=['Churn'])
    
    # Show sample
    print("\n   Sample of flipped rows:")
    flipped_count = 0
    for idx in random_indices[:5]:
        row = df_new.iloc[idx]
        if pd.notna(row['customerID']):
            print(f"     {row['customerID']}: Label='{row['churn_label']}'")
            flipped_count += 1
    if len(random_indices) > 5:
        print(f"     ... and {len(random_indices) - 5} more")
    
    # Show label statistics
    labeled = df_new['churn_label'].notna().sum()
    print(f"\n   Label statistics in new_data:")
    print(f"     - Total rows: {len(df_new)}")
    print(f"     - Rows with labels: {labeled}")
    print(f"     - 'Yes' labels: {(df_new['churn_label'] == 'Yes').sum()}")
    print(f"     - 'No' labels: {(df_new['churn_label'] == 'No').sum()}")

    
    # Create new_data table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS new_data (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            customerID TEXT UNIQUE,
            gender TEXT,
            SeniorCitizen INTEGER,
            Partner TEXT,
            Dependents TEXT,
            tenure INTEGER,
            PhoneService TEXT,
            MultipleLines TEXT,
            InternetService TEXT,
            OnlineSecurity TEXT,
            OnlineBackup TEXT,
            DeviceProtection TEXT,
            TechSupport TEXT,
            StreamingTV TEXT,
            StreamingMovies TEXT,
            Contract TEXT,
            PaperlessBilling TEXT,
            PaymentMethod TEXT,
            MonthlyCharges REAL,
            TotalCharges REAL,
            prediction INTEGER,
            probability REAL,
            churn_label TEXT,
            label_timestamp TIMESTAMP,
            created_at TIMESTAMP
        )
    ''')
    
    # Drop existing data if any and insert new
    cursor.execute("DELETE FROM new_data")
    
    
    df_new.to_sql('new_data', conn, if_exists='append', index=False)
    print(f"\n✅ Table 'new_data' created with {len(df_new)} rows")
    
    # =========================================================
    # TABLE 3: api_keys (empty - will be filled at startup)
    # =========================================================
    print("\n" + "="*60)
    print("STEP 4: Creating api_keys table")
    print("="*60)
    
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS api_keys (
            key_hash TEXT PRIMARY KEY,
            name TEXT,
            role TEXT,
            rate_limit INTEGER DEFAULT 100,
            is_active INTEGER DEFAULT 1,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    ''')
    
    # Clear existing keys
    cursor.execute("DELETE FROM api_keys")
    
    # Create dashboard API key
    
    raw_key = "dashboard"
    key_hash = hashlib.sha256(raw_key.encode()).hexdigest()
    
    cursor.execute('''
        INSERT INTO api_keys (key_hash, name, role, rate_limit, is_active, created_at)
        VALUES (?, ?, ?, ?, ?, ?)
    ''', (
        key_hash,
        "dashboard",
        "dashboard",
        1000,
        1,
        datetime.now().isoformat()
    ))
    
    print("✅ Table 'api_keys' created with default dashboard key")

        
    # =========================================================
    # SUMMARY
    # =========================================================
    conn.commit()
    conn.close()
    
    print("\n" + "="*60)
    print(" DATABASE CREATED SUCCESSFULLY!")
    print("="*60)
    print(f"📁 Database location: {DB_PATH}")
    print(f"📊 File size: {DB_PATH.stat().st_size / 1024:.1f} KB")
    print("\n📋 Summary:")
    print(f"   - raw_data: {len(df_raw)} rows (all original data)")
    print(f"   - new_data: {len(df_new)} rows (predictions + labels)")
    print(f"     • {(predictions == 1).sum()} predicted to churn")
    print(f"     • {(predictions == 0).sum()} predicted to stay")
    print(f"     • {labeled} rows have labels (flipped from original)")
    print(f"   - api_keys: 0 rows (will be added at startup)")
    print("\n🚀 Next step: Run 'uvicorn app.main:app --reload'")
    print("   You will be prompted to enter API keys")
    print("="*60)


if __name__ == "__main__":
    create_database()


STEP 1: Loading raw data
✅ Loaded 7043 rows from ..\data\raw\telco_churn.csv
✅ Table 'raw_data' created with 7043 rows

STEP 2: Creating new_data with 200 random rows
✅ Selected 200 random rows
   Shape before predictions: (200, 21)
📦 Loading model and making predictions...
✅ Model loaded from ..\models\full_churn_pipeline_cloud.pkl
   Shape after adding predictions: (200, 24)

STEP 3: Setting labels for all 200 rows
   Shape after adding label columns: (200, 26)
✅ Labels set for all 200 rows
   - 20 rows have FLIPPED labels
   - Final shape: (200, 26)

   Sample of flipped rows:
     2439-QKJUL: Label='Yes'
     5922-ABDVO: Label='No'
     3082-YVEKW: Label='Yes'
     0455-XFASS: Label='Yes'
     4707-YNOQA: Label='Yes'
     ... and 15 more

   Label statistics in new_data:
     - Total rows: 200
     - Rows with labels: 200
     - 'Yes' labels: 70
     - 'No' labels: 130

✅ Table 'new_data' created with 200 rows

STEP 4: Creating api_keys table
✅ Table 'api_keys' created with defaul